# Play Tennis Prediction using Naive Bayes

This notebook follows the handwritten exercise: create the Play Tennis dataset, inspect class counts, build frequency and conditional-probability tables, and classify a new instance using the **Naive Bayes / Maximum A Posteriori (MAP)** rule.

The model assumes that the features are conditionally independent given the class. This is the categorical version of Naive Bayes.

## 1. Import libraries and load the dataset

The CSV is stored beside this notebook, so the notebook can be run from the same folder.

In [ ]:
from pathlib import Path
import pandas as pd

DATA_FILE = Path("play_tennis.csv")
df = pd.read_csv(DATA_FILE)

display(df)
print(f"Dataset shape: {df.shape}")

## 2. Inspect the target classes

Count how many examples belong to each target class.

In [ ]:
class_counts = df["Play Tennis"].value_counts().sort_index()
print("Class counts:")
display(class_counts.to_frame(name="Count"))

print("\\nClass proportions:")
display((class_counts / len(df)).to_frame(name="Prior probability"))

## 3. Frequency tables for each feature

For every input feature, display a cross-tabulation against the target class. These are the counts used to calculate conditional probabilities.

In [ ]:
features = ["Outlook", "Temperature", "Humidity", "Wind"]
target = "Play Tennis"

for column in features:
    print(f"\\n{column}")
    display(pd.crosstab(df[column], df[target], margins=True))

## 4. Conditional probability tables

Each cell below is \`P(feature=value | Play Tennis=class)\`. The denominator is the number of training examples in that class.

In [ ]:
conditional_tables = {}

for feature in features:
    table = pd.crosstab(df[feature], df[target], normalize="columns")
    conditional_tables[feature] = table
    print(f"P({feature} | class)")
    display(table.round(4))

## 5. Define the new instance

The test instance from the notes is:

- Outlook = Sunny
- Temperature = Cool
- Humidity = High
- Wind = Strong

In [ ]:
new_instance = {
    "Outlook": "Sunny",
    "Temperature": "Cool",
    "Humidity": "High",
    "Wind": "Strong",
}

print("Instance to classify:")
print(new_instance)

# Validate feature names and categories before prediction.
for feature, value in new_instance.items():
    if feature not in features:
        raise KeyError(f"Unknown feature: {feature}")
    if value not in set(df[feature]):
        raise ValueError(f"Unknown value {value!r} for feature {feature!r}")

## 6. Calculate Naive Bayes probabilities

For each class, calculate:

\[P(C \mid X) \propto P(C) \prod_i P(x_i \mid C)\]

The notebook prints the prior, each conditional frequency, the likelihood product, and the unnormalized posterior score. Log probabilities are used internally to keep the calculation numerically stable.

In [ ]:
import math

classes = sorted(df[target].unique())
class_priors = df[target].value_counts(normalize=True).to_dict()

scores = {}
details = []

for cls in classes:
    class_rows = df[df[target] == cls]
    log_score = math.log(class_priors[cls])

    for feature, value in new_instance.items():
        count = int(((class_rows[feature] == value)).sum())
        total = len(class_rows)
        probability = count / total
        details.append({
            "Class": cls,
            "Feature": feature,
            "Value": value,
            "Frequency": count,
            "Class total": total,
            "Conditional probability": probability,
        })
        if probability == 0:
            log_score = float("-inf")
        elif log_score != float("-inf"):
            log_score += math.log(probability)

    scores[cls] = log_score

print("Conditional-frequency calculations:")
display(pd.DataFrame(details))

print("Log posterior scores (prior included, before normalization):")
for cls in classes:
    print(f"Play Tennis = {cls}: {scores[cls]:.6f}")

## 7. Normalize scores and make the MAP prediction

The largest posterior probability determines the predicted class. This is the Maximum A Posteriori (MAP) decision.

In [ ]:
finite_scores = [value for value in scores.values() if math.isfinite(value)]
if not finite_scores:
    raise ValueError("Every class received zero probability. Try Laplace smoothing.")

max_log_score = max(finite_scores)
unnormalized = {
    cls: (math.exp(score - max_log_score) if math.isfinite(score) else 0.0)
    for cls, score in scores.items()
}
normalizer = sum(unnormalized.values())
posterior = {cls: value / normalizer for cls, value in unnormalized.items()}

print("Posterior probabilities:")
for cls, probability in posterior.items():
    print(f"P(Play Tennis={cls} | X) = {probability:.6%}")

prediction = max(posterior, key=posterior.get)
print(f"\\nMAP prediction: Play Tennis = {prediction}")

## 8. Optional: predict another instance

Edit the values below and rerun this cell and the prediction cells if you want to test another combination.

In [ ]:
def predict_play_tennis(instance, data=df, smoothing=0):
    """Predict Yes/No with categorical Naive Bayes.

    smoothing=0 gives the unsmoothed probabilities used in the notes.
    Set smoothing=1 for Laplace (add-one) smoothing.
    """
    feature_names = ["Outlook", "Temperature", "Humidity", "Wind"]
    target_name = "Play Tennis"
    classes_local = sorted(data[target_name].unique())
    log_scores = {}

    for cls in classes_local:
        class_rows = data[data[target_name] == cls]
        class_total = len(class_rows)
        log_probability = math.log(class_total / len(data))

        for feature in feature_names:
            value = instance[feature]
            if value not in set(data[feature]):
                raise ValueError(f"Unknown value {value!r} for {feature!r}")

            count = int((class_rows[feature] == value).sum())
            category_count = data[feature].nunique()
            probability = (count + smoothing) / (class_total + smoothing * category_count)

            if probability == 0:
                log_probability = float("-inf")
                break
            log_probability += math.log(probability)

        log_scores[cls] = log_probability

    best_log = max(log_scores.values())
    weights = {
        cls: math.exp(value - best_log) if math.isfinite(value) else 0.0
        for cls, value in log_scores.items()
    }
    total_weight = sum(weights.values())
    probabilities = {cls: weight / total_weight for cls, weight in weights.items()}
    return max(probabilities, key=probabilities.get), probabilities

example = {
    "Outlook": "Sunny",
    "Temperature": "Cool",
    "Humidity": "High",
    "Wind": "Strong",
}
label, probabilities = predict_play_tennis(example)
print("Prediction:", label)
print("Posterior probabilities:", probabilities)

## 9. Summary

For the example instance, the model predicts **No**. The class probabilities are obtained by multiplying the class prior by the four feature likelihoods, then normalizing the two class scores. The optional function also supports Laplace smoothing for unseen feature/class combinations.